In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - pdfplumber: opens PDFs and exposes per-page text and table extraction
#   - csv: writes extracted records to a CSV file
#
# Requirements: pip install pdfplumber

import csv

import pdfplumber

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your PDF path and extraction options here before running any section below.
#
# input_path   - Path to the PDF file you want to extract from
# mode         - 'text' to extract raw page text, 'table' to extract structured rows
# pages        - Page range string (e.g. '1,3,5-7') or None to process all pages
# table_index  - Which table per page to extract, 0-indexed (table mode only)
# output_path  - Where to save the resulting CSV

# --- Text mode example ---
input_path   = 'report.pdf'
mode         = 'text'
pages        = None       # None = all pages, or e.g. '1-3'
table_index  = 0          # only used in table mode
output_path  = 'output.csv'

# --- Table mode example (uncomment to use) ---
# input_path   = 'data.pdf'
# mode         = 'table'
# pages        = '1-5'
# table_index  = 0
# output_path  = 'table.csv'

print(f"Config ready. Input: {input_path} | Mode: {mode} | Pages: {pages or 'all'}")

In [ ]:
# %%
# SECTION 3: load_pdf()
# -----------------------
# Opens the PDF at input_path using pdfplumber and returns a PDF object.
# Prints the total page count and the metadata from the first page so you
# can confirm the right file was opened before running extraction.
# Stores the result in 'pdf' — all subsequent sections use this object.

import pathlib

def load_pdf(path: str):
    if not pathlib.Path(path).exists():
        raise FileNotFoundError(f"File not found: '{path}'")
    return pdfplumber.open(path)

pdf = load_pdf(input_path)
print(f"Opened '{input_path}': {len(pdf.pages)} page(s)")
print(f"Metadata: {pdf.metadata}")

In [ ]:
# %%
# SECTION 4: parse_page_numbers()
# ---------------------------------
# Converts the 'pages' config string into a sorted list of 0-based page indices.
# Supports comma-separated values and ranges, e.g. '1,3,5-7' → [0, 2, 4, 5, 6].
# When pages is None, returns all page indices. Prints the resolved list so you
# can verify before running extraction.

def parse_page_numbers(pages_arg, total_pages):
    if pages_arg is None:
        return list(range(total_pages))
    indices = set()
    for part in pages_arg.split(','):
        part = part.strip()
        if '-' in part:
            start, end = part.split('-', 1)
            indices.update(range(int(start) - 1, int(end)))
        else:
            indices.add(int(part) - 1)
    return sorted(i for i in indices if 0 <= i < total_pages)

page_indices = parse_page_numbers(pages, len(pdf.pages))
print(f"Processing {len(page_indices)} page(s): {[i + 1 for i in page_indices]}")

In [ ]:
# %%
# SECTION 5: Inspect pages
# -------------------------
# Previews the first 300 characters of extracted text from each selected page.
# Use this to confirm the PDF contains selectable text (not scanned images)
# and that the right pages are selected before running full extraction.
# If pages show no text, the PDF may be image-based and require OCR.

print(f"Previewing {min(3, len(page_indices))} page(s):\n")
for i in page_indices[:3]:
    text = pdf.pages[i].extract_text()
    preview = text[:300].strip() if text else '(no text found)'
    print(f"--- Page {i + 1} ---")
    print(preview)
    print()

In [ ]:
# %%
# SECTION 6: extract_text() / extract_tables()
# ----------------------------------------------
# Runs the extraction defined by 'mode' in Section 2.
#
# text mode:  Extracts raw text from each page. Returns one dict per page
#             with keys 'page' (1-based int) and 'text' (str).
#
# table mode: Extracts a structured table from each page. Uses the first row
#             as the header and flattens all rows into a list of dicts.
#             A 'page' column is prepended to each row.
#             Pages with no table at table_index are silently skipped.

def extract_text(pdf, page_indices):
    records = []
    for i in page_indices:
        text = pdf.pages[i].extract_text()
        if text and text.strip():
            records.append({'page': i + 1, 'text': text.strip()})
    return records

def extract_tables(pdf, page_indices, table_index):
    records = []
    for i in page_indices:
        tables = pdf.pages[i].extract_tables()
        if not tables or table_index >= len(tables):
            continue
        rows = [r for r in tables[table_index] if any(c for c in r)]
        if len(rows) < 2:
            continue
        header = [str(c) if c is not None else '' for c in rows[0]]
        for row in rows[1:]:
            record = {'page': i + 1}
            for col, val in zip(header, row):
                record[col] = val if val is not None else ''
            records.append(record)
    return records

if mode == 'text':
    records = extract_text(pdf, page_indices)
    fieldnames = ['page', 'text']
else:
    records = extract_tables(pdf, page_indices, table_index)
    fieldnames = list(records[0].keys()) if records else ['page']

print(f"Extracted {len(records)} record(s) in '{mode}' mode.")

In [ ]:
# %%
# SECTION 7: Inspect result
# --------------------------
# Prints the column names and first 5 records so you can review the extracted
# data before writing it to disk. If the output looks wrong, adjust mode,
# pages, or table_index in Section 2 and re-run from Section 6.

print(f"Columns: {fieldnames}")
print(f"\nFirst 5 records:")
for r in records[:5]:
    print(r)

In [ ]:
# %%
# SECTION 8: write_csv()
# -----------------------
# Writes the extracted records to a CSV at output_path (set in Section 2).
# Run this last, after confirming the Section 7 preview looks correct.

def write_csv(records, fieldnames, output_path):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"Wrote {len(records)} records to '{output_path}'")

if records:
    write_csv(records, fieldnames, output_path)
else:
    print('No records to write. Check your configuration and re-run Section 6.')

pdf.close()